# Prepare ML Dataset

This notebook collects pre-computed per-basin feature CSVs (from `00_full_pipeline.ipynb`),
applies hard-negative filtering and stratified subsampling, and saves the master dataset.

**Input:** `data/results/{basin}/full_features.csv` (one per basin)  
**Output:** `data/results/master_dataset_v2.csv`

## 1. Setup & Configuration

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from channel_heads import filter_hard_negatives
from channel_heads.config import RESULTS_DIR

print(f"Results directory: {RESULTS_DIR}")

Results directory: /Users/guypi/Projects/channel-heads/data/results


In [2]:
# === CONFIGURATION ===

# Hard negative filtering thresholds
HARD_NEG_MAX_L_RATIO = 3.0      # Max L_sum ratio vs median positive
HARD_NEG_MAX_DIST_RATIO = 5.0   # Max distance ratio vs median positive

# Negative subsampling
NEGATIVE_RATIO = 3.0   # Target negative:positive ratio
RANDOM_SEED = 42

# Output
MASTER_OUTPUT_PATH = RESULTS_DIR / "master_dataset_v2.csv"

print("Configuration:")
print(f"  Hard negative max_L_ratio: {HARD_NEG_MAX_L_RATIO}")
print(f"  Hard negative max_dist_ratio: {HARD_NEG_MAX_DIST_RATIO}")
print(f"  Target neg:pos ratio: {NEGATIVE_RATIO}:1")
print(f"  Output: {MASTER_OUTPUT_PATH}")

Configuration:
  Hard negative max_L_ratio: 3.0
  Hard negative max_dist_ratio: 5.0
  Target neg:pos ratio: 3.0:1
  Output: /Users/guypi/Projects/channel-heads/data/results/master_dataset_v2.csv


## 2. Load Per-Basin Feature CSVs

Load all `full_features.csv` files generated by `00_full_pipeline.ipynb`.

In [3]:
# Discover per-basin full_features.csv files
basin_csvs = {}
for basin_dir in sorted(RESULTS_DIR.iterdir()):
    if basin_dir.is_dir() and basin_dir.name != "experiments":
        csv_path = basin_dir / "full_features.csv"
        if csv_path.exists():
            basin_csvs[basin_dir.name] = csv_path

print(f"Found {len(basin_csvs)} basin feature files:\n")
for basin, path in basin_csvs.items():
    df_tmp = pd.read_csv(path)
    n_pos = (df_tmp["y"] == 1).sum() if "y" in df_tmp.columns else 0
    print(f"  {basin:20s}: {len(df_tmp):>7,} rows  ({n_pos:,} touching)")

if len(basin_csvs) == 0:
    raise FileNotFoundError(
        "No full_features.csv files found. Run 00_full_pipeline.ipynb first."
    )

Found 17 basin feature files:

  calnalpine          :       7 rows  (4 touching)
  daqing              :      13 rows  (10 touching)
  finisterre          :   1,821 rows  (415 touching)
  humboldt            :      13 rows  (8 touching)
  inyo                :      15 rows  (7 touching)
  kammanasie          :      19 rows  (6 touching)
  luliang             :      73 rows  (33 touching)
  panamint            :      60 rows  (22 touching)
  sakhalin            :      71 rows  (28 touching)
  sierramadre         :   2,577 rows  (576 touching)
  sierranevadaspain   :      48 rows  (23 touching)
  taiwan              :   5,471 rows  (1,054 touching)
  toano               :      27 rows  (12 touching)
  troodos             :     171 rows  (60 touching)
  tsugaru             :      12 rows  (6 touching)
  vallefertil         :      93 rows  (46 touching)
  yoro                :       5 rows  (5 touching)


In [4]:
# Combine all basins
all_dfs = []
for basin, path in basin_csvs.items():
    df_basin = pd.read_csv(path)
    if "basin" not in df_basin.columns:
        df_basin["basin"] = basin
    all_dfs.append(df_basin)

df_combined = pd.concat(all_dfs, ignore_index=True)

print(f"Combined dataset: {df_combined.shape[0]:,} rows x {df_combined.shape[1]} columns")
print(f"\nColumns: {list(df_combined.columns)}")
print(f"\nClass balance (raw):")
print(f"  y=1 (touching):     {(df_combined['y'] == 1).sum():>7,}")
print(f"  y=0 (not touching): {(df_combined['y'] == 0).sum():>7,}")

Combined dataset: 10,496 rows x 23 columns

Columns: ['outlet', 'confluence', 'head_1', 'head_2', 'touching', 'contact_px', 'size1_px', 'size2_px', 'skipped_prefilter', 'y', 'L_1', 'L_2', 'delta_L', 'orientation_diff_deg', 'headhead_dist_m', 'headhead_dist_norm', 'apex_angle_deg', 'strahler_order_diff', 'proximity_mean_m', 'proximity_max_m', 'proximity_profile_norm', 'qc_flags', 'basin']

Class balance (raw):
  y=1 (touching):       2,315
  y=0 (not touching):   8,181


## 3. Missing Values & QC Summary

In [5]:
# Feature completeness check
feature_cols = [
    # Coupling
    "contact_px", "size1_px", "size2_px",
    # Asymmetry
    "L_1", "L_2", "delta_L",
    # Geometric
    "orientation_diff_deg",
    "headhead_dist_m", "headhead_dist_norm",
    "apex_angle_deg", "strahler_order_diff",
]

print("=== Missing Values ===\n")
for col in feature_cols:
    if col in df_combined.columns:
        n_miss = df_combined[col].isna().sum()
        pct = n_miss / len(df_combined) * 100
        marker = " ⚠" if pct > 5 else ""
        print(f"  {col:24s}: {n_miss:>7,} ({pct:5.1f}%){marker}")
    else:
        print(f"  {col:24s}: MISSING COLUMN")

# QC flags
if "qc_flags" in df_combined.columns:
    has_flags = df_combined["qc_flags"].notna() & (df_combined["qc_flags"] != "")
    print(f"\nRows with QC flags: {has_flags.sum():,} ({has_flags.mean()*100:.1f}%)")

=== Missing Values ===

  contact_px              :       0 (  0.0%)
  size1_px                :   7,368 ( 70.2%) ⚠
  size2_px                :   7,368 ( 70.2%) ⚠
  L_1                     :       0 (  0.0%)
  L_2                     :       0 (  0.0%)
  delta_L                 :       0 (  0.0%)
  orientation_diff_deg    :       0 (  0.0%)
  headhead_dist_m         :       0 (  0.0%)
  headhead_dist_norm      :       0 (  0.0%)
  apex_angle_deg          :       0 (  0.0%)
  strahler_order_diff     :       0 (  0.0%)

Rows with QC flags: 171 (1.6%)


## 4. Hard Negative Filtering

Remove trivially non-touching pairs (very different scale or very far apart).

In [6]:
print(f"Applying hard negative filtering...")
print(f"  max_L_ratio:    {HARD_NEG_MAX_L_RATIO}")
print(f"  max_dist_ratio: {HARD_NEG_MAX_DIST_RATIO}")

df_filtered = filter_hard_negatives(
    df_combined,
    max_L_ratio=HARD_NEG_MAX_L_RATIO,
    max_dist_ratio=HARD_NEG_MAX_DIST_RATIO,
)

n_removed = len(df_combined) - len(df_filtered)
print(f"\n  Before: {len(df_combined):>7,} rows")
print(f"  After:  {len(df_filtered):>7,} rows")
print(f"  Removed: {n_removed:,} trivial negatives ({n_removed / len(df_combined) * 100:.1f}%)")
print(f"\n  Class balance after filtering:")
print(f"    y=1: {(df_filtered['y'] == 1).sum():>6,}")
print(f"    y=0: {(df_filtered['y'] == 0).sum():>6,}")

Applying hard negative filtering...
  max_L_ratio:    3.0
  max_dist_ratio: 5.0

  Before:  10,496 rows
  After:    5,922 rows
  Removed: 4,574 trivial negatives (43.6%)

  Class balance after filtering:
    y=1:  2,315
    y=0:  3,607


## 5. Stratified Negative Subsampling

Subsample negatives per basin to achieve target ratio while preserving all positives.

In [7]:
def stratified_subsample_negatives(
    df: pd.DataFrame,
    target_ratio: float = 3.0,
    random_state: int = 42,
) -> pd.DataFrame:
    """Subsample negatives per basin to target neg:pos ratio, keeping all positives."""
    if df.empty:
        return df

    rng = np.random.default_rng(random_state)
    positives = df[df["y"] == 1].copy()
    negatives = df[df["y"] == 0].copy()

    n_pos = len(positives)
    n_neg_target = int(n_pos * target_ratio)

    print(f"  Positives:          {n_pos:,}")
    print(f"  Negatives (current): {len(negatives):,}")
    print(f"  Negatives (target):  {n_neg_target:,}")

    if len(negatives) <= n_neg_target:
        print(f"  -> Keeping all negatives (already below target)")
        return df

    # Stratified sampling by basin
    basin_neg_counts = negatives.groupby("basin").size()
    basin_proportions = basin_neg_counts / basin_neg_counts.sum()
    basin_targets = (basin_proportions * n_neg_target).round().astype(int)

    # Adjust rounding to hit exact target
    diff = n_neg_target - basin_targets.sum()
    for basin in basin_neg_counts.sort_values(ascending=False).index:
        if diff == 0:
            break
        if diff > 0:
            basin_targets[basin] += 1
            diff -= 1
        elif diff < 0 and basin_targets[basin] > 0:
            basin_targets[basin] -= 1
            diff += 1

    sampled = []
    for basin, n_target in basin_targets.items():
        basin_negs = negatives[negatives["basin"] == basin]
        if len(basin_negs) <= n_target:
            sampled.append(basin_negs)
        else:
            idx = rng.choice(len(basin_negs), size=n_target, replace=False)
            sampled.append(basin_negs.iloc[idx])

    df_sampled_neg = pd.concat(sampled, ignore_index=True)
    result = pd.concat([positives, df_sampled_neg], ignore_index=True)
    result = result.sort_values(["basin", "outlet", "confluence"], ignore_index=True)

    actual_ratio = len(df_sampled_neg) / n_pos if n_pos > 0 else 0
    print(f"  -> Final: {n_pos:,} pos + {len(df_sampled_neg):,} neg (ratio={actual_ratio:.2f}:1)")
    return result


print(f"Subsampling negatives (target ratio: {NEGATIVE_RATIO}:1)...\n")
df_master = stratified_subsample_negatives(df_filtered, NEGATIVE_RATIO, RANDOM_SEED)

Subsampling negatives (target ratio: 3.0:1)...

  Positives:          2,315
  Negatives (current): 3,607
  Negatives (target):  6,945
  -> Keeping all negatives (already below target)


## 6. Save Master Dataset

In [8]:
# Save
df_master.to_csv(MASTER_OUTPUT_PATH, index=False)

file_size_mb = MASTER_OUTPUT_PATH.stat().st_size / 1e6
print(f"Saved master dataset:")
print(f"  Path:  {MASTER_OUTPUT_PATH}")
print(f"  Shape: {df_master.shape}")
print(f"  Size:  {file_size_mb:.2f} MB")

Saved master dataset:
  Path:  /Users/guypi/Projects/channel-heads/data/results/master_dataset_v2.csv
  Shape: (5922, 23)
  Size:  1.44 MB


## 7. Dataset Summary

In [9]:
print("=" * 60)
print("MASTER DATASET SUMMARY")
print("=" * 60)

print(f"\nTotal rows:    {len(df_master):,}")
print(f"Total columns: {len(df_master.columns)}")
print(f"Basins:        {df_master['basin'].nunique()}")

# Class balance
n_pos = (df_master["y"] == 1).sum()
n_neg = (df_master["y"] == 0).sum()
print(f"\nClass balance:")
print(f"  y=1 (touching):     {n_pos:>6,} ({n_pos / len(df_master) * 100:.1f}%)")
print(f"  y=0 (not touching): {n_neg:>6,} ({n_neg / len(df_master) * 100:.1f}%)")
print(f"  Ratio (neg:pos):    {n_neg / max(n_pos, 1):.2f}:1")

# Per-basin
print(f"\nPer-basin counts:")
basin_summary = df_master.groupby("basin").agg(
    total=("y", "count"),
    touching=("y", "sum"),
    ratio=("y", "mean"),
).sort_values("total", ascending=False)
basin_summary["touching"] = basin_summary["touching"].astype(int)
print(basin_summary.to_string())

MASTER DATASET SUMMARY

Total rows:    5,922
Total columns: 23
Basins:        17

Class balance:
  y=1 (touching):      2,315 (39.1%)
  y=0 (not touching):  3,607 (60.9%)
  Ratio (neg:pos):    1.56:1

Per-basin counts:
                   total  touching     ratio
basin                                       
taiwan              2729      1054  0.386222
sierramadre         1517       576  0.379697
finisterre          1090       415  0.380734
troodos              154        60  0.389610
vallefertil           80        46  0.575000
luliang               73        33  0.452055
sakhalin              68        28  0.411765
panamint              52        22  0.423077
sierranevadaspain     48        23  0.479167
toano                 27        12  0.444444
kammanasie            19         6  0.315789
inyo                  15         7  0.466667
humboldt              13         8  0.615385
daqing                13        10  0.769231
tsugaru               12         6  0.500000
calnalpine      

# Preview
df_master.head(10)